In [0]:
%python
df = spark.read.csv("/Volumes/new_catalog/new_schema/bronze_sales_etl/ecommerce_sales_etl_dataset.csv")
display(df)

In [0]:
df.printSchema()

In [0]:
df = spark.read.csv("/Volumes/new_catalog/new_schema/bronze_sales_etl/ecommerce_sales_etl_dataset.csv",header=True,inferSchema=True)
display(df)

In [0]:
df.printSchema()

In [0]:
df.count()

In [0]:
df.columns

In [0]:
#df.write.format("delta").saveAsTable("new_catalog.new_schema.bronze_sales")


In [0]:


display(spark.sql("select * from new_catalog.new_schema.bronze_sales"))

In [0]:
spark.sql("select count(*) as total_records from new_catalog.new_schema.bronze_sales").show()

In [0]:
display(spark.sql("select count(*) from new_catalog.new_schema.bronze_sales"))

In [0]:
from pyspark.sql.functions import col,sum,when 

null_counts = df.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in df.columns])
display(null_counts)

In [0]:
duplicates = df.groupBy(df.columns).count().filter("count > 1")
display(duplicates)

In [0]:
df.groupBy("OrderID") .count() .filter(col("count") > 1) .orderBy(col("count").desc()) .show()

In [0]:
df.groupBy("CustomerID").count().filter(col("count")>1).orderBy(col("count")).show()

In [0]:
df.filter(col("Quantity")<=0).show()

In [0]:
df.filter(col("UnitPrice")<=0).show()

In [0]:
df.groupBy("Category").count().orderBy("Category").show()

In [0]:
df.filter(col("CustomerID").isNull()).count()

In [0]:
df.filter(col("OrderID") == "ORD00984").show(truncate=False)

In [0]:
duplicate = df.groupBy(df.columns).count().filter(col("count")>1)
display(duplicate)

********************************************************************************************************************************************

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
silver_df = df

In [0]:
#Remove exact duplicate rows
silver_df = silver_df.dropDuplicates()

In [0]:
#Remove invalid quantities
silver_df = silver_df.filter(col("Quantity")>0)

In [0]:
#Remove invalid prices
silver_df = silver_df.filter(col("UnitPrice") > 0)

In [0]:
silver_df = silver_df.withColumn("Category",initcap(trim(col("Category"))))
'''
OUTPUT:-
electronics → Electronics
Electronics → Electronics
accessories → Accessories
'''

In [0]:
#Handle CustomerID NULLs
silver_df = silver_df.withColumn("CustomerID",when(col("CustomerID").isNull(), "UNKNOWN").otherwise(col("CustomerID")))


In [0]:
#Calculate Sales Amount
silver_df = silver_df.withColumn("GrossAmount",col("Quantity")*col("UnitPrice"))

In [0]:
#Calculate Discount Amount
silver_df = silver_df.withColumn("DiscountAmount",col("GrossAmount")*col("DiscountPercent")/100)

In [0]:
#Calculate Net Sales
silver_df = silver_df.withColumn("NetSales",col("GrossAmount")-col("DiscountAmount"))

In [0]:
display(silver_df)

In [0]:
display(df)

In [0]:
print("Bronze records:",df.count())
print("Silver records:",silver_df.count())

In [0]:
df.filter(col("OrderID") == "ORD00984").show(truncate=False)

In [0]:
df.groupBy(df.columns).count() .filter(col("count") > 1).show(truncate=False)

In [0]:
from pyspark.sql.functions import col, trim, initcap

silver_df = silver_df.withColumn(
    "Category",
    initcap(trim(col("Category")))
)

display(silver_df.select("Category").distinct())

In [0]:
string_columns = [
    "OrderID",
    "CustomerID",
    "ProductID",
    "ProductName",
    "Category",
    "PaymentMethod",
    "City",
    "State",
    "Country",
    "OrderStatus"
]

for c in string_columns:
    silver_df = silver_df.withColumn(
        c,
        trim(col(c))
    )

display(silver_df)

In [0]:
from pyspark.sql.types import IntegerType, DoubleType

silver_df = silver_df \
    .withColumn("Quantity", col("Quantity").cast(IntegerType())) \
    .withColumn("UnitPrice", col("UnitPrice").cast(DoubleType())) \
    .withColumn("DiscountPercent", col("DiscountPercent").cast(DoubleType()))

display(silver_df)

In [0]:
from pyspark.sql.functions import round

silver_df = silver_df.withColumn(
    "GrossAmount",
    round(col("Quantity") * col("UnitPrice"), 2)
)

display(
    silver_df.select(
        "OrderID",
        "ProductName",
        "Quantity",
        "UnitPrice",
        "GrossAmount"
    )
)

In [0]:
#Calculate Discount Amount

silver_df = silver_df.withColumn(
    "DiscountAmount",
    round(
        col("GrossAmount") * col("DiscountPercent") / 100,
        2
    )
)

display(
    silver_df.select(
        "OrderID",
        "GrossAmount",
        "DiscountPercent",
        "DiscountAmount"
    )
)

In [0]:
#Calculate NetAmount
silver_df = silver_df.withColumn(
    "NetAmount",
    round(
        col("GrossAmount") - col("DiscountAmount"),
        2
    )
)

display(
    silver_df.select(
        "OrderID",
        "Quantity",
        "UnitPrice",
        "GrossAmount",
        "DiscountPercent",
        "DiscountAmount",
        "NetAmount"
    )
)

In [0]:
#Validate the new columns
silver_df.select(
    "GrossAmount",
    "DiscountAmount",
    "NetAmount"
).describe().show()

In [0]:
silver_df.filter(
    (col("GrossAmount") < 0) |
    (col("DiscountAmount") < 0) |
    (col("NetAmount") < 0)
).show()

In [0]:
#Check Silver again
print("Silver records:", silver_df.count())

In [0]:
display(
    silver_df.select(
        "OrderID",
        "OrderDate",
        "CustomerID",
        "ProductName",
        "Category",
        "Quantity",
        "UnitPrice",
        "DiscountPercent",
        "GrossAmount",
        "DiscountAmount",
        "NetAmount",
        "PaymentMethod",
        "City",
        "State",
        "OrderStatus"
    )
)

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("new_catalog.new_schema.silver_sales")

In [0]:
spark.sql(""" SELECT COUNT(*) AS total_records FROM new_catalog.new_schema.silver_sales""").show()

In [0]:
silver_df = spark.table("new_catalog.new_schema.silver_sales")

display(silver_df)

In [0]:
from pyspark.sql.functions import sum, count, avg, round

gold_category_sales = (
    silver_df
    .groupBy("Category")
    .agg(
        round(sum("GrossAmount"), 2).alias("GrossSales"),
        round(sum("DiscountAmount"), 2).alias("TotalDiscount"),
        round(sum("NetAmount"), 2).alias("NetSales"),
        sum("Quantity").alias("TotalQuantity"),
        count("OrderID").alias("TotalOrders"),
        round(avg("NetAmount"), 2).alias("AverageOrderValue")
    )
    .orderBy(col("NetSales").desc())
)

display(gold_category_sales)

In [0]:
# Check the Category Gold table

display(
    spark.sql("""
        SELECT *
        FROM new_catalog.new_schema.gold_category_sales
        ORDER BY NetSales DESC
    """)
)

In [0]:
gold_category_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("new_catalog.new_schema.gold_category_sales")

In [0]:
spark.sql("""SELECT * FROM new_catalog.new_schema.gold_category_sales ORDER BY NetSales DESC""").show()

In [0]:
#Create Daily Sales Gold Table
gold_daily_sales = (
    silver_df
    .groupBy("OrderDate")
    .agg(
        round(sum("GrossAmount"), 2).alias("GrossSales"),
        round(sum("DiscountAmount"), 2).alias("TotalDiscount"),
        round(sum("NetAmount"), 2).alias("NetSales"),
        sum("Quantity").alias("TotalQuantity"),
        count("OrderID").alias("TotalOrders")
    )
    .orderBy("OrderDate")
)

display(gold_daily_sales)

In [0]:
gold_daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("new_catalog.new_schema.gold_daily_sales")

In [0]:
#Create Product Sales Gold Table
gold_product_sales = (
    silver_df
    .groupBy("ProductID", "ProductName", "Category")
    .agg(
        sum("Quantity").alias("TotalQuantity"),
        round(sum("GrossAmount"), 2).alias("GrossSales"),
        round(sum("DiscountAmount"), 2).alias("TotalDiscount"),
        round(sum("NetAmount"), 2).alias("NetSales"),
        count("OrderID").alias("TotalOrders")
    )
    .orderBy(col("NetSales").desc())
)

display(gold_product_sales)

In [0]:
gold_product_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("new_catalog.new_schema.gold_product_sales")

In [0]:
#Create City Sales Gold Table
gold_city_sales = (
    silver_df
    .groupBy("City", "State")
    .agg(
        round(sum("NetAmount"), 2).alias("NetSales"),
        sum("Quantity").alias("TotalQuantity"),
        count("OrderID").alias("TotalOrders"),
        round(avg("NetAmount"), 2).alias("AverageOrderValue")
    )
    .orderBy(col("NetSales").desc())
)

display(gold_city_sales)

In [0]:
gold_city_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("new_catalog.new_schema.gold_city_sales")

In [0]:
#Create Customer Sales Gold Table
gold_customer_sales = (
    silver_df
    .filter(col("CustomerID").isNotNull())
    .groupBy("CustomerID")
    .agg(
        count("OrderID").alias("TotalOrders"),
        sum("Quantity").alias("TotalQuantity"),
        round(sum("NetAmount"), 2).alias("TotalSpent"),
        round(avg("NetAmount"), 2).alias("AverageOrderValue")
    )
    .orderBy(col("TotalSpent").desc())
)

display(gold_customer_sales)

In [0]:
gold_customer_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("new_catalog.new_schema.gold_customer_sales")

In [0]:
from pyspark.sql.functions import (
    sum,
    countDistinct,
    avg,
    col,
    round
)

gold_summary = silver_df.select(
    round(
        sum(
            col("Quantity") *
            col("UnitPrice") *
            (1 - col("DiscountPercent") / 100)
        ),
        2
    ).alias("Total_Revenue"),

    countDistinct("OrderID").alias("Total_Orders"),

    countDistinct("CustomerID").alias("Total_Customers"),

    sum("Quantity").alias("Total_Quantity"),

    round(
        sum(
            col("Quantity") *
            col("UnitPrice") *
            (1 - col("DiscountPercent") / 100)
        ) / countDistinct("OrderID"),
        2
    ).alias("Average_Order_Value")
)

display(gold_summary)

In [0]:
gold_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("new_catalog.new_schema.gold_sales_summary")

In [0]:
spark.sql("""SELECT * FROM new_catalog.new_schema.gold_sales_summary """).show()

In [0]:
#Validate the Gold tables
gold_tables = [
    "gold_daily_sales",
    "gold_category_sales",
    "gold_product_sales",
    "gold_customer_sales",
    "gold_city_sales",
    "gold_sales_summary"
]

for table in gold_tables:
    count = spark.table(f"new_catalog.new_schema.{table}").count()
    print(f"{table}: {count} records")

In [0]:
#Check the Category Gold table
display(spark.sql("""SELECT * FROM new_catalog.new_schema.gold_category_sales ORDER BY Revenue DESC """)
)

In [0]:
# Dashboard Query 1 - Daily Sales

daily_sales_dashboard = spark.sql("""
    SELECT
        OrderDate,
        NetSales,
        TotalOrders
    FROM new_catalog.new_schema.gold_daily_sales
    ORDER BY OrderDate
""")

display(daily_sales_dashboard)

In [0]:
# Dashboard Query 2 - Sales by Category

category_sales_dashboard = spark.sql("""
    SELECT
        Category,
        NetSales,
        TotalOrders,
        TotalQuantity
    FROM new_catalog.new_schema.gold_category_sales
    ORDER BY NetSales DESC
""")

display(category_sales_dashboard)

In [0]:
# Dashboard Query 3 - Top Products

product_sales_dashboard = spark.sql("""
    SELECT
        ProductName,
        Category,
        NetSales,
        TotalQuantity,
        TotalOrders
    FROM new_catalog.new_schema.gold_product_sales
    ORDER BY NetSales DESC
    LIMIT 10
""")

display(product_sales_dashboard)

In [0]:
# Dashboard KPI Summary

dashboard_kpi = spark.sql("""
SELECT
    Total_Revenue,
    Total_Orders,
    Total_Customers,
    Total_Quantity,
    Average_Order_Value
FROM new_catalog.new_schema.gold_sales_summary
""")

display(dashboard_kpi)

In [0]:
# Daily Sales Dashboard Data

daily_sales_dashboard = spark.sql("""
SELECT
    OrderDate,
    NetSales,
    TotalOrders
FROM new_catalog.new_schema.gold_daily_sales
ORDER BY OrderDate
""")

display(daily_sales_dashboard)